# PRODUCTS - INCREMENTAL LOAD WITH AUTO LOADER

In [0]:
%run /Workspace/Users/rms181800@gmail.com/AZURE_B3_PROJECT_AUG/FUNCTIONS/functions

In [0]:
from pyspark.sql.functions import current_timestamp

df_stream = spark.readStream.format("cloudFiles").option("cloudFiles.format", "csv").option("header", "true").option("inferSchema", "true").option("cloudFiles.schemaLocation", "/Volumes/retail_project_b3/bronze/_checkpoints/products_schema").load("/Volumes/retail_project_b3/landing/raw_data/olist_products_dataset.csv").withColumn("ingest_ts", current_timestamp())

query = df_stream.writeStream.format("delta").option("checkpointLocation", "/Volumes/retail_project_b3/bronze/_checkpoints/products").outputMode("append").trigger(availableNow=True).start("/Volumes/retail_project_b3/bronze/products/")
query.awaitTermination()
print(f"✅ Processed {query.lastProgress.get('numInputRows', 0) if query.lastProgress else 0} records")